# New-Product Extraction from SEC Form 8-K Filings (NER + LLM)

**Course:** CAP5619 AI in Finance  
**GitHub:** `<your repo URL>`

This notebook drives `extract_8k_products.py`. Pipeline: S&P 500 universe → SEC EDGAR 8-K filings (Items 7.01/8.01) → press-release exhibits (EX-99.x) → keyword pre-filter → spaCy + rule-based NER → local LLM via Ollama (JSON schema output) → validation → CSV.

## 0. Setup
```bash
pip install -r requirements.txt
python -m spacy download en_core_web_sm
ollama pull llama3.1:8b      # or: ollama pull deepseek-r1:8b
ollama serve                  # if not already running
```

In [ ]:
import os, requests
from pathlib import Path
import pandas as pd
import extract_8k_products as ex

# SEC fair-access policy: identify yourself (name + email)
os.environ.setdefault('SEC_USER_AGENT', 'Your Name your.email@ucf.edu')

# Check that Ollama is up and which models are installed
print([m['name'] for m in requests.get('http://localhost:11434/api/tags', timeout=5).json()['models']])

## 1. NER demo on the assignment example
spaCy's small model finds the organization but often mislabels product names (e.g. as PERSON). The rule-based pattern (capitalised phrase after a launch verb) recovers them, and the LLM makes the final call.

In [ ]:
nlp = ex.load_nlp()
s = 'Apple Inc. has announced the launch of its latest MacBook Pro, featuring a new M3 processor.'
print('spaCy entities :', [(e.text, e.label_) for e in nlp(s).ents])
print('rule-based     :', ex.rule_based_products(s))
passage, cands = ex.run_ner(nlp, s)
print('candidates     :', cands)

## 2. Configure the run
`max_documents` counts filings actually sent to the LLM (those that pass the launch-keyword pre-filter). Re-running resumes where it stopped: filings already in `extraction_audit.csv` are skipped.

In [ ]:
cfg = ex.Config(
    user_agent=os.environ['SEC_USER_AGENT'],
    start_date='2025-11-01', end_date='2026-10-31',
    max_documents=100,
    backend='ollama', model='llama3.1:8b',
    tickers=None,            # e.g. ['AAPL','NVDA','MSFT'] for a quick test
    filing_time_format='date',
    out_dir=Path('output'),
)
cfg

## 3. Quick single-filing test (optional)

In [ ]:
test_cfg = ex.Config(**{**cfg.__dict__, 'tickers': ['NVDA'], 'max_documents': 2, 'out_dir': Path('output_test')})
ex.run(test_cfg)

## 4. Full run

In [ ]:
%%time
df = ex.run(cfg)
df.head(20)

## 5. Results and checks

In [ ]:
df = pd.read_csv(cfg.out_csv)
audit = pd.read_csv(cfg.audit_csv)

print('Filings scanned          :', len(audit))
print(audit['status'].value_counts().to_string())
llm_docs = audit['status'].str.startswith('llm').sum()
hit = (audit['status'] == 'llm_products').sum()
print(f'\nDocuments sent to LLM    : {llm_docs}')
print(f'Documents with a product : {hit} ({hit / max(llm_docs, 1):.0%})')
print('Product rows             :', len(df), '| companies:', df['stock_name'].nunique())
print('Max description length   :', df['product_description'].str.len().max())
assert (df['product_description'].str.len() < 180).all()
rej = audit['rejected'].dropna()
print('Rejected by validation   :', rej.str.count(';').add(1).sum())

In [ ]:
print(ex.write_pipe_table(cfg.out_csv, cfg.table_txt))

## 6. Manual accuracy check
Sample 20 rows, open each `url` from the audit file and mark whether the product and description are correct. Report the precision in the write-up.

In [ ]:
sample = df.sample(min(20, len(df)), random_state=42).merge(
    audit[['ticker', 'filing_date', 'url']].rename(columns={'ticker': 'stock_name', 'filing_date': 'filing_time'}),
    on=['stock_name', 'filing_time'], how='left').drop_duplicates()
sample['correct'] = ''
sample.to_csv('output/manual_review.csv', index=False)
sample